# 02 — Schema Validation (Canonical Contracts)

**Fecha:** 2026-02-02  
**Objetivo:** Definir y validar el *schema canonical* para OHLCV 1m y Quotes (p95) usando el slice AABA 2019-01.

**Inputs:**
- Manifest: `data/manifests/r2_slice_AABA_2019_01.json`
- Cache local: `data/cache/...`

**Outputs:**
- Contrato de schema (documentado en este notebook)
- Métricas de calidad específicas del schema (crossed markets, duplicados, nulos)
- Reglas/decisiones para normalización y tratamiento


In [1]:
import sys, os, time
from pathlib import Path

NOTEBOOK_ID = "02_schema_validation"
def detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for cand in candidates:
        if (cand / "data" / "manifests").exists() and (cand / "notebooks" / "01_data_integrity").exists():
            return cand
    return cwd

PROJECT_ROOT = detect_project_root().parents[1]
os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(PROJECT_ROOT)

print("python:", sys.executable)
print("cwd:", os.getcwd())
t0 = time.time()
time.sleep(0.2)
print("ok, elapsed:", round(time.time() - t0, 3))


python: C:\TSIS_Data\v1\backtest_SmallCaps\backtest\Scripts\python.exe
cwd: C:\TSIS_Data\v1\backtest_SmallCaps
ok, elapsed: 0.201


In [2]:
import os
import json
import subprocess
from pathlib import Path
import polars as pl
import sys

NOTEBOOK_ID = "02_schema_validation"

def detect_project_root() -> Path:
    # 1) explicit override
    env_root = os.getenv("PROJECT_ROOT")
    if env_root:
        p = Path(env_root).resolve()
        if (p / "notebooks" / "01_data_integrity").exists() and (p / "src" / "data").exists():
            return p

    # 2) known local default for this project
    known = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps")
    if known.exists() and (known / "notebooks" / "01_data_integrity").exists():
        return known

    # 3) search upward from cwd, preferring stronger repo markers
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    best = None
    best_score = -1
    for cand in candidates:
        score = 0
        if (cand / "notebooks" / "01_data_integrity").exists():
            score += 1
        if (cand / "data" / "manifests").exists():
            score += 1
        if (cand / "src" / "data" / "catalog.py").exists():
            score += 2
        if (cand / "scripts" / "build_manifest.py").exists():
            score += 2
        if cand.name.lower() == "backtest_smallcaps":
            score += 3
        if score > best_score:
            best_score = score
            best = cand

    return best if best is not None else cwd

PROJECT_ROOT = detect_project_root()

manifest_name = "r2_slice_AABA_2019_01.json"
manifest_candidates = [
    PROJECT_ROOT / "data" / "manifests" / manifest_name,
    Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\manifests") / manifest_name,
]
MANIFEST_PATH = next((m for m in manifest_candidates if m.exists()), manifest_candidates[0])

os.environ["DATA_CACHE_DIR"] = r"C:\TSIS_Data\data"
CACHE_DIR = Path(os.environ["DATA_CACHE_DIR"]).resolve()
RUNS_DIR = Path(os.getenv("RUNS_DIR", PROJECT_ROOT / "runs")).resolve()

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Cache dir:", CACHE_DIR)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest no existe: {MANIFEST_PATH}")
if not CACHE_DIR.exists():
    resp = input("Cache dir no existe. Materializar desde R2 con r2_sync? [y/N] ")
    if resp.strip().lower() == "y":
        sync_cmd = [
            sys.executable, str(PROJECT_ROOT / "scripts" / "r2_sync.py"),
            "--manifest", str(MANIFEST_PATH),
        ]
        print("SYNC:", " ".join(sync_cmd))
        subprocess.check_call(sync_cmd, cwd=str(PROJECT_ROOT), env=env)
    if not CACHE_DIR.exists():
        raise FileNotFoundError(f"Cache dir no existe: {CACHE_DIR} (define DATA_CACHE_DIR o materializa con r2_sync)")


Manifest: C:\TSIS_Data\v1\backtest_SmallCaps\data\manifests\r2_slice_AABA_2019_01.json
Cache dir: C:\TSIS_Data\data


In [3]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
objs = manifest["objects"]

ohlcv_keys = [o["key"] for o in objs if o["dataset"] == "ohlcv_intraday_1m"]
quote_keys = [o["key"] for o in objs if o["dataset"] == "quotes_p95"]

print("OHLCV files:", len(ohlcv_keys))
print("Quote files:", len(quote_keys))
print("Example OHLCV:", ohlcv_keys[0])
print("Example Quote:", quote_keys[0])


OHLCV files: 1
Quote files: 21
Example OHLCV: ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/minute.parquet
Example Quote: quotes_p95/AABA/year=2019/month=01/day=02/quotes.parquet


In [4]:
ohlcv_path = CACHE_DIR / ohlcv_keys[0]
df_ohlcv = pl.read_parquet(ohlcv_path, n_rows=5)

print("OHLCV columns:", df_ohlcv.columns)
print("OHLCV dtypes:", df_ohlcv.dtypes)
df_ohlcv


OHLCV columns: ['volume', 'vwap', 'open', 'close', 'high', 'low', 't', 'transactions', 'timestamp', 'ticker', 'date', 'minute']
OHLCV dtypes: [Int64, Float64, Float64, Float64, Float64, Float64, Int64, Int64, Datetime(time_unit='ms', time_zone=None), String, Date, String]


volume,vwap,open,close,high,low,t,transactions,timestamp,ticker,date,minute
i64,f64,f64,f64,f64,f64,i64,i64,datetime[ms],str,date,str
1000,57.0,57.0,57.0,57.0,57.0,1546435560000,5,2019-01-02 13:26:00,"""AABA""",2019-01-02,"""2019-01-02 13:26"""
188,57.16,57.16,57.16,57.16,57.16,1546436820000,1,2019-01-02 13:47:00,"""AABA""",2019-01-02,"""2019-01-02 13:47"""
11114,57.94,57.94,57.94,57.94,57.94,1546437000000,1,2019-01-02 13:50:00,"""AABA""",2019-01-02,"""2019-01-02 13:50"""
22833,56.8276,56.78,56.72,56.97,56.72,1546439400000,111,2019-01-02 14:30:00,"""AABA""",2019-01-02,"""2019-01-02 14:30"""
21724,56.7218,56.74,56.59,56.84,56.57,1546439460000,159,2019-01-02 14:31:00,"""AABA""",2019-01-02,"""2019-01-02 14:31"""


In [5]:
quote_path = CACHE_DIR / quote_keys[0]
df_q = pl.read_parquet(quote_path, n_rows=5)

print("Quotes columns:", df_q.columns)
print("Quotes dtypes:", df_q.dtypes)
df_q


Quotes columns: ['ask_exchange', 'ask_price', 'ask_size', 'bid_exchange', 'bid_price', 'bid_size', 'conditions', 'participant_timestamp', 'sequence_number', 'timestamp', 'tape', 'indicators']
Quotes dtypes: [Int64, Float64, Int64, Int64, Float64, Int64, List(Int64), Int64, Int64, Int64, Int64, List(Int64)]


ask_exchange,ask_price,ask_size,bid_exchange,bid_price,bid_size,conditions,participant_timestamp,sequence_number,timestamp,tape,indicators
i64,f64,i64,i64,f64,i64,list[i64],i64,i64,i64,i64,list[i64]
18,57.17,200,15,56.33,700,[1],1546439400000189852,303792,1546439400060097104,3,null
12,57.17,1000,15,56.33,700,[1],1546439400297882330,305701,1546439400297899484,3,null
12,57.17,1000,2,56.38,300,[1],1546439400319048147,306018,1546439400319064390,3,null
12,57.17,2000,2,56.38,300,[1],1546439400463833200,307975,1546439400463852123,3,null
7,57.17,5200,2,56.38,300,[1],1546439400685353000,310328,1546439400685553615,3,null


## Canonical schema contract

### OHLCV 1m (canonical)
- ts: int64 (epoch ns/us/ms) o datetime64[ns] (definir política)
- open, high, low, close: float64
- volume: int64

### Quotes p95 (canonical)
- ts: int64 (epoch …) tomado de `timestamp` (por defecto)
- bid: float64 (desde bid_price)
- ask: float64 (desde ask_price)
- bid_size, ask_size: int64
- (otros campos se conservan como metadata, no requeridos para checks básicos)


**Normalización en notebook: bid/ask/ts**

In [6]:
def normalize_quotes(df: pl.DataFrame) -> pl.DataFrame:
    out = df
    if "bid" not in out.columns and "bid_price" in out.columns:
        out = out.rename({"bid_price": "bid"})
    if "ask" not in out.columns and "ask_price" in out.columns:
        out = out.rename({"ask_price": "ask"})
    # canonical timestamp column name
    if "ts" not in out.columns and "timestamp" in out.columns:
        out = out.rename({"timestamp": "ts"})
    return out

df_qn = normalize_quotes(pl.read_parquet(quote_path))

[i for i in df_qn.columns]

['ask_exchange',
 'ask',
 'ask_size',
 'bid_exchange',
 'bid',
 'bid_size',
 'conditions',
 'participant_timestamp',
 'sequence_number',
 'ts',
 'tape',
 'indicators']

**Medir crossed markets por día**

Esto cuantifica el WARN:

In [7]:
import json

# Cargar manifest
m = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
objs = m.get("objects", m)

# Extraer keys normalizadas
all_keys = [
    o["key"] if isinstance(o, dict) else o
    for o in objs
]

def normalize_key(k: str) -> str:
    k = k.lstrip("/").replace("\\", "/")
    if k.startswith("s3://"):
        k = k.split("/", 3)[3]
    return k

all_keys = [normalize_key(k) for k in all_keys]

# Filtrar SOLO quotes
quotes_keys = [k for k in all_keys if k.startswith("quotes_p95/")]

print("Total manifest keys:", len(all_keys))
print("Quotes keys:", len(quotes_keys))
print("Example quote key:", quotes_keys[0] if quotes_keys else None)


Total manifest keys: 22
Quotes keys: 21
Example quote key: quotes_p95/AABA/year=2019/month=01/day=02/quotes.parquet


In [8]:
import polars as pl
from pathlib import Path

def crossed_metrics_for_key_lazy(cache_dir: Path, key: str) -> dict:
    """
    Calcula metricas de 'crossed market' (bid > ask) para un parquet de quotes
    usando scan_parquet (lazy) para escalar.

    Devuelve dict con:
      - key
      - total_rows
      - crossed_rows
      - crossed_ratio
    """
    p = cache_dir / key
    if not p.exists():
        return {"key": key, "total_rows": None, "crossed_rows": None, "crossed_ratio": None, "missing": True}

    lf = pl.scan_parquet(str(p))
    schema = lf.collect_schema().names()

    bid_col = pl.col("bid_price").alias("bid") if "bid_price" in schema else pl.col("bid")
    ask_col = pl.col("ask_price").alias("ask") if "ask_price" in schema else pl.col("ask")

    lf = (
        lf.select([bid_col, ask_col])
        .with_columns([
            pl.col("bid").cast(pl.Float64),
            pl.col("ask").cast(pl.Float64),
        ])
        .with_columns([
            (pl.col("bid") > pl.col("ask")).alias("crossed")
        ])
    )

    agg = (
        lf.select([
            pl.len().alias("total_rows"),
            pl.sum("crossed").cast(pl.Int64).alias("crossed_rows"),
        ])
        .collect()
    )

    total = int(agg["total_rows"][0])
    crossed = int(agg["crossed_rows"][0]) if agg["crossed_rows"][0] is not None else 0
    ratio = (crossed / total) if total > 0 else 0.0

    return {
        "key": key,
        "total_rows": total,
        "crossed_rows": crossed,
        "crossed_ratio": ratio,
        "missing": False,
    }

# --- Aplicacion a todas las keys de quotes del manifest ---
# Asume que ya tienes: CACHE_DIR y quotes_keys (lista de keys)
rows = [crossed_metrics_for_key_lazy(CACHE_DIR, k) for k in quotes_keys]

df_cross = pl.DataFrame(rows).sort("key")

# Resumen rapido
summary = (
    df_cross
    .filter(~pl.col("missing"))
    .select([
        pl.sum("total_rows").alias("total_rows"),
        pl.sum("crossed_rows").alias("crossed_rows"),
        (pl.sum("crossed_rows") / pl.sum("total_rows")).alias("crossed_ratio_weighted"),
        pl.mean("crossed_ratio").alias("crossed_ratio_mean"),
        pl.len().alias("n_files"),
    ])
)

display(df_cross.head(10))
display(summary)


key,total_rows,crossed_rows,crossed_ratio,missing
str,i64,i64,f64,bool
"""quotes_p95/AABA/year=2019/mont…",159598,6,0.000038,false
"""quotes_p95/AABA/year=2019/mont…",50000,0,0.0,false
"""quotes_p95/AABA/year=2019/mont…",50000,0,0.0,false
"""quotes_p95/AABA/year=2019/mont…",161264,0,0.0,false
"""quotes_p95/AABA/year=2019/mont…",158881,11,0.000069,false
"""quotes_p95/AABA/year=2019/mont…",215085,3,0.000014,false
"""quotes_p95/AABA/year=2019/mont…",50000,0,0.0,false
"""quotes_p95/AABA/year=2019/mont…",117594,1,0.000009,false
"""quotes_p95/AABA/year=2019/mont…",127937,9,0.00007,false


total_rows,crossed_rows,crossed_ratio_weighted,crossed_ratio_mean,n_files
i64,i64,f64,f64,u32
3359402,229,0.000068,0.000051,21


## Decisión: tratamiento de crossed markets

Se observa `bid > ask` en el slice (AABA 2019-01) con ratio global medido.

Política (v1):
- Mantener como WARN en validación.
- Para backtesting: aplicar “clamp” en el modelo de ejecución:
  - bid = min(bid, ask)
  - ask = max(ask, bid)
  - y registrar cuántas filas fueron corregidas.


**Guardar resultados en runs para trazabilidad**

In [9]:
from datetime import datetime
from pathlib import Path

RUNS_DIR = Path(os.getenv("RUNS_DIR", "./runs")).resolve()
RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

OUT = RUNS_DIR / "data_quality" / NOTEBOOK_ID / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

df_cross.write_parquet(OUT / "crossed_market_by_day.parquet")

summary = {
    "manifest": str(MANIFEST_PATH),
    "cache_dir": str(CACHE_DIR),
    "run_id": RUN_ID,
}
(Path(OUT) / "schema_review_meta.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved to:", OUT)


Saved to: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\schema_validation\AABA_2019_01\20260211_181149
